# MODUL PRAKTIKUM BIG DATA
## Pertemuan 13 — Model Persistence & Integrasi Big Data Analitik End-to-End

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 13 |
| **Topik** | Menyimpan & Memuat Model ML, Studi Kasus Integrasi Menyeluruh (MongoDB + Spark + MLlib + HDFS) |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-12 selesai — pertemuan ini mengintegrasikan **seluruh** kemampuan yang telah dipelajari sepanjang semester |

---

### Cara Menggunakan Modul Ini

**Tidak ada instalasi baru.** Seluruh modul dikerjakan di Jupyter Notebook. Pertemuan ini bersifat **kulminatif** — kita akan menggabungkan MongoDB (Pertemuan 9), Spark DataFrame & ETL (Pertemuan 4-7), dan Spark MLlib (Pertemuan 11-12) menjadi **satu alur kerja utuh**, sekaligus mempelajari satu kemampuan praktis yang selama ini terlewat: **menyimpan model yang sudah dilatih agar dapat dipakai ulang di kemudian hari** tanpa perlu melatih ulang dari awal.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11**, **Hadoop 3.4.3**, **MongoDB 8.0** — seluruh stack yang telah dipasang sejak awal semester, tidak ada perubahan versi.


---
## Recap Komprehensif: Pertemuan 9-12

Sebelum masuk ke topik baru, berikut rangkuman kemampuan yang telah anda kuasai sejak UTS:

| Pertemuan | Kemampuan Utama |
|---|---|
| **9** | Basis data NoSQL, instalasi MongoDB, CRUD dengan PyMongo, integrasi MongoDB → Spark DataFrame |
| **10** | Konsep batch vs streaming, Spark Structured Streaming, agregasi berbasis window waktu |
| **11** | Pengenalan Spark MLlib, feature engineering (`StringIndexer`, `VectorAssembler`), regresi linear |
| **12** | `Pipeline`, `OneHotEncoder`, klasifikasi (Logistic Regression, Random Forest), evaluasi model |

Satu hal yang **belum pernah kita bahas**: bagaimana jika model yang sudah susah payah dilatih pada Pertemuan 11-12 ingin dipakai lagi besok, minggu depan, atau oleh aplikasi lain — **tanpa perlu melatih ulang dari nol setiap kali**? Inilah topik pembuka pertemuan ini.

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 13, mahasiswa mampu:
1. Menyimpan (*save*) model Spark MLlib yang sudah dilatih ke disk/HDFS.
2. Memuat kembali (*load*) model tersimpan pada sesi yang benar-benar baru, tanpa melatih ulang.
3. Merancang dan menjalankan alur kerja Big Data analitik yang mengintegrasikan sumber data NoSQL, pemrosesan Spark, dan model Machine Learning tersimpan menjadi satu pipeline utuh.


---
## 13.1 Mengapa Model Persistence Penting?

Bayangkan skenario nyata: anda melatih model klasifikasi pada Pertemuan 12 yang membutuhkan waktu beberapa menit (atau bisa berjam-jam untuk dataset sungguhan berskala besar). Jika `SparkSession` ditutup — entah karena Jupyter di-restart, VM dimatikan, atau aplikasi lain yang memakai model ini berjalan di waktu berbeda — **seluruh hasil pelatihan akan hilang** jika tidak disimpan secara eksplisit, dan anda harus mengulang seluruh proses `fit()` dari awal.

**Model persistence** menyimpan model (termasuk seluruh tahap `Pipeline` — indexer, encoder, hingga model klasifikasi/regresi) ke dalam berkas, sehingga dapat dimuat kembali kapan saja pada sesi Spark yang benar-benar baru.

### 13.1.1 Persiapan: Melatih Ulang Model Sederhana (Melanjutkan Pertemuan 12)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Pertemuan13-ModelPersistence") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(21)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 100000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
})
data["total_pendapatan"] = data["unit_terjual"] * data["harga_satuan"]
threshold = data["total_pendapatan"].median()
prob_tinggi = 1 / (1 + np.exp(-(data["total_pendapatan"] - threshold) / 100000))
data["label"] = (np.random.rand(n) < prob_tinggi).astype(int)

df = spark.createDataFrame(data)

# Pipeline identik dengan pola Pertemuan 12
idx_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
assembler = VectorAssembler(inputCols=["unit_terjual", "harga_satuan", "kategori_idx"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[idx_kategori, assembler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

print("Model berhasil dilatih pada sesi ini.")

### 13.1.2 Menyimpan Model (`save`)

In [ ]:
# write().overwrite().save() -- menyimpan SELURUH tahap pipeline (indexer, assembler, model)
# "overwrite()" memastikan tidak terjadi error jika folder tujuan sudah ada dari percobaan sebelumnya
path_model = "model_klasifikasi_tersimpan"
model.write().overwrite().save(path_model)

print(f"Model berhasil disimpan ke folder '{path_model}'")
!ls -la {path_model}

Perhatikan Spark menyimpan model sebagai **folder** (bukan satu berkas tunggal) berisi dua sub-folder: `stages` (parameter tiap tahap pipeline) dan `metadata` (informasi versi & konfigurasi) — mirip prinsip penyimpanan Parquet yang terpartisi pada Pertemuan 6.

### 13.1.3 Mensimulasikan "Sesi Baru" dan Memuat Kembali Model

Untuk benar-benar membuktikan model dapat dipakai ulang tanpa pelatihan ulang, mari **tutup SparkSession saat ini**, lalu buat SparkSession **baru** — mensimulasikan situasi seperti membuka Jupyter di lain waktu.

In [ ]:
spark.stop()
print("SparkSession lama ditutup (mensimulasikan Jupyter yang baru dibuka kembali).")

In [ ]:
from pyspark.ml import PipelineModel

# Membuat SparkSession BARU -- seolah-olah ini adalah sesi kerja yang sama sekali berbeda
spark = SparkSession.builder \
    .appName("Pertemuan13-SesiBaru") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# PipelineModel.load() -- memuat kembali model yang telah disimpan, TANPA proses fit() sama sekali
model_dimuat = PipelineModel.load(path_model)

print("Model berhasil dimuat kembali pada sesi Spark yang benar-benar baru!")
print("Jumlah tahap pipeline:", len(model_dimuat.stages))

### 13.1.4 Menggunakan Model yang Dimuat untuk Prediksi Data Baru

In [ ]:
# Data transaksi BARU yang belum pernah dilihat model sama sekali
data_baru = spark.createDataFrame(pd.DataFrame({
    "unit_terjual": [3, 12, 7],
    "harga_satuan": [25000, 100000, 50000],
    "kategori": ["Fashion", "Elektronik", "Makanan"],
}))

prediksi_baru = model_dimuat.transform(data_baru)
prediksi_baru.select("unit_terjual", "harga_satuan", "kategori", "prediction", "probability").show(truncate=False)

Model berhasil menghasilkan prediksi **tanpa satu baris kode `fit()` pun** dijalankan pada sesi ini — inilah kekuatan model persistence: melatih sekali, pakai berkali-kali, bahkan oleh aplikasi/script yang sama sekali berbeda selama dapat mengakses folder model tersimpan tersebut (termasuk jika disimpan di HDFS dan diakses dari berbagai node cluster).

> **Untuk produksi real:** folder model biasanya disimpan di HDFS (bukan disk lokal) — persis seperti pola `hdfs://localhost:9000/...` yang telah anda pakai sejak Pertemuan 4 — agar dapat diakses dari node manapun dalam cluster, bukan hanya dari mesin tempat model dilatih.


---
## 13.2 Studi Kasus: Pipeline Analitik End-to-End

**Konteks:** "Kopi Nusantara" (skenario yang telah menemani kita sejak Pertemuan 7) kini memiliki alur kerja mingguan berikut: (1) data pesanan terbaru dicatat ke **MongoDB** oleh sistem kasir tiap cabang, (2) setiap akhir pekan, tim data melakukan **ekstraksi & analisis** menggunakan Spark, (3) sebuah **model prediksi** (yang telah dilatih & disimpan sebelumnya) diterapkan untuk mengidentifikasi pesanan berisiko mendapat ulasan buruk, dan (4) hasil akhirnya disimpan sebagai laporan.

Mari kita bangun **seluruh alur ini** menggunakan kombinasi teknik dari Pertemuan 9, 11-12, dan 4-6.

### Langkah 1: Menyisipkan Data "Mentah" ke MongoDB (Mensimulasikan Sistem Kasir)

In [ ]:
from pymongo import MongoClient
import random

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara_p13"]
koleksi_pesanan = db["pesanan_mingguan"]
koleksi_pesanan.delete_many({})  # membersihkan data lama agar percobaan konsisten

random.seed(303)
n_pesanan = 500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]

# Catatan: sengaja menggunakan modul `random` bawaan Python (bukan np.random) di sini --
# np.random.choice() menghasilkan tipe numpy (mis. numpy.str_) yang TIDAK dapat diserialisasi
# langsung oleh PyMongo ke format BSON. Modul `random` bawaan menghasilkan tipe Python murni (str, int)
# yang aman disisipkan ke MongoDB.
dokumen_pesanan = [
    {
        "nama_menu": random.choice(menu_list),
        "cabang": random.choice(cabang_list),
        "qty": random.randint(1, 5),
        "waktu_tunggu_menit": random.randint(2, 24),
    }
    for _ in range(n_pesanan)
]
koleksi_pesanan.insert_many(dokumen_pesanan)

print(f"{koleksi_pesanan.count_documents({})} dokumen pesanan berhasil disisipkan ke MongoDB.")

### Langkah 2: Mengekstraksi Data dari MongoDB ke Spark DataFrame (pola Pertemuan 9)

In [ ]:
dokumen_mongo = list(koleksi_pesanan.find({}, {"_id": 0}))  # {"_id": 0} artinya kolom _id tidak perlu diikutsertakan
df_pandas = pd.DataFrame(dokumen_mongo)

df_pesanan = spark.createDataFrame(df_pandas)
df_pesanan.show(5)
print("Total pesanan diekstraksi dari MongoDB:", df_pesanan.count())

### Langkah 3: Melatih & Menyimpan Model Prediksi Risiko Ulasan Buruk (pola Pertemuan 12-13.1)

*(Pada praktik nyata, tahap ini biasanya sudah dilakukan sebelumnya dan modelnya tinggal dimuat — namun kita latih ulang di sini secara singkat agar modul ini dapat dijalankan mandiri dari awal hingga akhir.)*

In [ ]:
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.classification import RandomForestClassifier

# Label simulasi: risiko ulasan buruk lebih tinggi saat waktu tunggu lama
prob_buruk = 1 / (1 + np.exp(-(df_pandas["waktu_tunggu_menit"] - 12) / 4))
df_pandas["label_risiko_buruk"] = (np.random.rand(len(df_pandas)) < prob_buruk).astype(int)
df_latih = spark.createDataFrame(df_pandas)

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
ohe = OneHotEncoder(inputCols=["menu_idx", "cabang_idx"], outputCols=["menu_vec", "cabang_vec"])
assembler_risiko = VectorAssembler(inputCols=["menu_vec", "cabang_vec", "qty", "waktu_tunggu_menit"], outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label_risiko_buruk", numTrees=25)

pipeline_risiko = Pipeline(stages=[idx_menu, idx_cabang, ohe, assembler_risiko, rf])
train_r, test_r = df_latih.randomSplit([0.8, 0.2], seed=42)
model_risiko = pipeline_risiko.fit(train_r)

path_model_risiko = "model_risiko_ulasan"
model_risiko.write().overwrite().save(path_model_risiko)
print("Model risiko ulasan buruk berhasil dilatih & disimpan.")

### Langkah 4: Menerapkan Model pada Data Pesanan Real (dari MongoDB)

In [ ]:
model_risiko_dimuat = PipelineModel.load(path_model_risiko)

hasil_prediksi = model_risiko_dimuat.transform(df_pesanan)
hasil_prediksi = hasil_prediksi.withColumnRenamed("prediction", "prediksi_risiko_buruk")

hasil_prediksi.select("nama_menu", "cabang", "qty", "waktu_tunggu_menit", "prediksi_risiko_buruk").show(10)

In [ ]:
from pyspark.sql.functions import col, count

print("Ringkasan jumlah pesanan berdasarkan prediksi risiko:")
hasil_prediksi.groupBy("prediksi_risiko_buruk").agg(count("nama_menu").alias("jumlah_pesanan")).show()

print("\nRingkasan risiko per cabang:")
hasil_prediksi.groupBy("cabang").agg(
    count("nama_menu").alias("total_pesanan"),
).show()

ringkasan_risiko_cabang = hasil_prediksi.groupBy("cabang", "prediksi_risiko_buruk").agg(
    count("nama_menu").alias("jumlah")
).orderBy("cabang", "prediksi_risiko_buruk")
ringkasan_risiko_cabang.show()

### Langkah 5: Menyimpan Laporan Akhir sebagai Parquet (pola Pertemuan 6)

In [ ]:
hasil_prediksi.write.mode("overwrite").partitionBy("cabang").parquet("laporan_risiko_pesanan")

print("Laporan akhir berhasil disimpan sebagai Parquet, terpartisi per cabang.")
!ls laporan_risiko_pesanan

Anda baru saja menjalankan **satu alur kerja Big Data analitik yang utuh**, menggabungkan hampir seluruh kemampuan yang dipelajari sepanjang semester:

- **Pertemuan 9** — Data mentah tersimpan & diekstraksi dari MongoDB
- **Pertemuan 4-6** — Diproses sebagai Spark DataFrame, disimpan sebagai Parquet terpartisi
- **Pertemuan 11-12** — Model klasifikasi dilatih dengan `Pipeline`
- **Pertemuan 13 (hari ini)** — Model disimpan & dimuat kembali untuk dipakai berkali-kali tanpa pelatihan ulang

Inilah gambaran nyata bagaimana komponen-komponen yang terlihat terpisah di tiap pertemuan sebenarnya **saling terhubung** dalam satu ekosistem Big Data yang utuh di dunia kerja.

---

## Menutup Koneksi & SparkSession

In [ ]:
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup.")

---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` (Sub-bab 13.1.1) sebelum mengerjakan latihan berikut. Pastikan MongoDB (`sudo systemctl status mongod`) tetap aktif.

In [ ]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan13").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
print("Siap untuk latihan.")

**Soal 1.** Muat kembali model pada `path_model` (`model_klasifikasi_tersimpan` dari Sub-bab 13.1) menggunakan `PipelineModel.load()`, lalu buat **5 data transaksi rekaan baru** dan tampilkan hasil prediksinya.

In [ ]:
# Jawaban Soal 1 di sini


**Soal 2.** Buka koleksi MongoDB `kopi_nusantara_p13.pesanan_mingguan` dari Sub-bab 13.2, gunakan PyMongo untuk menghitung **rata-rata `waktu_tunggu_menit`** langsung melalui MongoDB Aggregation (petunjuk: cari operator `$group` dan `$avg` pada dokumentasi MongoDB — mirip konsep `groupBy().agg()` di Spark, namun dalam sintaks MongoDB).

In [ ]:
# Jawaban Soal 2 di sini


**Soal 3.** Muat kembali `model_risiko_ulasan` (Sub-bab 13.2 Langkah 3), lalu simpan **hanya baris dengan prediksi risiko buruk (label 1.0)** ke folder Parquet baru bernama `pesanan_berisiko_tinggi` (tanpa partisi).

In [ ]:
# Jawaban Soal 3 di sini


**Soal 4 (Refleksi menyeluruh).** Sebagai penutup rangkaian Pertemuan 9-13, jelaskan dalam 4-5 kalimat: bagaimana MongoDB, Spark (batch & ML), dan HDFS/Parquet **saling melengkapi** dalam satu alur kerja Big Data analitik yang realistis? Gunakan istilah dari seluruh pertemuan yang relevan (skema fleksibel, distributed processing, model persistence, columnar storage). Tulis jawaban pada markdown cell di bawah ini.

*(Tulis jawaban di sini — jangan lupa menutup koneksi & SparkSession setelah selesai berlatih)*


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 14 dimulai**.
> **Sifat tugas:** individu.
> Tugas ini mengintegrasikan **hampir seluruh** kemampuan praktikum sejak Pertemuan 9 — kerjakan secara bertahap, jangan menunda hingga mendekati tenggat waktu.

### Konteks / Skenario

Platform e-commerce (skenario yang menemani kita sejak Pertemuan 2) ingin membangun sistem **deteksi dini transaksi berisiko dibatalkan pelanggan** (*cancellation risk*). Data transaksi mentah dicatat di MongoDB oleh sistem pemesanan, dan tim data bertugas melatih, menyimpan, serta menerapkan model prediksi secara berkala.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas13_[NPM]_[Nama Lengkap].ipynb`**, lalu kerjakan bagian **A sampai F** berikut:

---

**A. Menyisipkan Data ke MongoDB** *(bobot 10%)*

Buat database baru `ecommerce_p13`, koleksi `transaksi_mentah`. Sisipkan **minimal 800 dokumen** transaksi dummy dengan field: `kategori`, `kota`, `unit_terjual`, `harga_satuan`, `metode_pembayaran`, dan `estimasi_pengiriman_hari` (angka 1-14). Buat label `dibatalkan` (0/1) yang probabilitasnya **dipengaruhi** oleh `estimasi_pengiriman_hari` (semakin lama estimasi, semakin besar kemungkinan dibatalkan) — gunakan pola pembuatan data probabilistik seperti pada Sub-bab 13.2.

> **Perhatian teknis:** Gunakan modul `random` bawaan Python (`random.choice()`, `random.randint()`), **bukan** `np.random.choice()`, saat membangun dokumen satu per satu dalam list comprehension untuk disisipkan ke MongoDB — tipe data numpy (mis. `numpy.str_`) tidak dapat diserialisasi langsung oleh PyMongo. Lihat kembali catatan pada Sub-bab 13.2 Langkah 1.

**B. Ekstraksi ke Spark DataFrame** *(bobot 10%)*

Ekstraksi seluruh dokumen dari MongoDB menjadi Spark DataFrame, tampilkan `printSchema()` dan jumlah barisnya.

**C. Melatih Model & Menyimpannya** *(bobot 25%)*

Bangun `Pipeline` (StringIndexer + OneHotEncoder + VectorAssembler + `RandomForestClassifier`, `labelCol="dibatalkan"`) menggunakan fitur `kategori`, `kota`, `metode_pembayaran`, `unit_terjual`, `harga_satuan`, dan `estimasi_pengiriman_hari`. Latih model (split 80:20, `seed=99`), evaluasi dengan **accuracy** dan **AUC-ROC**, lalu **simpan** model ke folder `model_pembatalan_tersimpan`.

**D. Simulasi "Minggu Berikutnya"** *(bobot 20%)*

Tutup `SparkSession`, buat **SparkSession baru**, lalu muat kembali model dari folder `model_pembatalan_tersimpan` menggunakan `PipelineModel.load()` — **tanpa melatih ulang**. Buat **10 transaksi baru** (boleh langsung sebagai Spark DataFrame atau disisipkan dulu ke MongoDB lalu diekstraksi ulang mengikuti pola bagian A-B), terapkan model yang dimuat untuk memprediksi risiko pembatalannya.

**E. Menyimpan Laporan Akhir** *(bobot 20%)*

Simpan hasil prediksi bagian D sebagai Parquet ke folder `laporan_risiko_pembatalan`, dipartisi berdasarkan `kategori`. Verifikasi dengan membaca kembali & menampilkan `count()`-nya.

**F. Kesimpulan** *(bobot 15%)*

Tulis pada markdown cell (**minimal 120 kata**): jelaskan alur kerja end-to-end yang telah anda bangun (dari MongoDB hingga laporan akhir), dan berikan **satu rekomendasi bisnis konkret** bagi tim operasional berdasarkan pola risiko pembatalan yang ditemukan model (mis. terkait `estimasi_pengiriman_hari`).

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas13_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan MongoDB aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- Bagian D **wajib** menunjukkan bukti nyata bahwa `SparkSession` benar-benar ditutup dan dibuat ulang (bukan hanya memuat model pada sesi yang sama) — inilah inti pembelajaran model persistence pada pertemuan ini.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Data ke MongoDB | Minimal 800 dokumen tersisip dengan label yang dipengaruhi fitur secara logis | 10% |
| B. Ekstraksi | Data berhasil diekstraksi & dikonversi ke Spark DataFrame dengan benar | 10% |
| C. Pelatihan & Penyimpanan | Pipeline benar, evaluasi masuk akal, model tersimpan | 25% |
| D. Simulasi Sesi Baru | SparkSession baru terbukti dibuat ulang, model dimuat tanpa fit() ulang | 20% |
| E. Laporan Akhir | Hasil tersimpan sebagai Parquet terpartisi & terverifikasi | 20% |
| F. Kesimpulan | Alur kerja dijelaskan runtut, rekomendasi bisnis relevan & berdasarkan data | 15% |
